<a href="https://colab.research.google.com/github/alearecuest/anyoneai-exercises-sprint_3/blob/main/03_lora_llm_fine_tuning_sentiment_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Notebook Overview**

This notebook teaches **parameter-efficient fine-tuning (PEFT)** using **LoRA** to adapt a Large Language Model for sentiment classification. It shows how to train modern LLMs even on modest hardware.

### **What You Will Learn**

* How LoRA works and why it is efficient
* Configuring PEFT adapters for transformer-based LLMs
* Applying LoRA to fine-tune a model on a sentiment dataset
* Training, validation, and evaluation workflows
* Comparing LoRA performance to zero-shot and full fine-tuning

### **Key Concepts Covered**

* Parameter-efficient training
* Low-Rank Adaptation (LoRA)
* Memory-efficient optimization
* Practical LLM fine-tuning for real-world tasks



## 1. Import Libraries

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType, PeftModel
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

## 2. Load IMDb Dataset

We'll use the same dataset as notebooks 1 and 2 for direct comparison.

In [ ]:
# Load IMDb dataset - same size as notebook 1 for fair comparison
from datasets import DatasetDict

full_dataset = load_dataset("imdb")

# Shuffle and take balanced subset
train_dataset = full_dataset["train"].shuffle(seed=42).select(range(1000))
test_dataset = full_dataset["test"].shuffle(seed=42).select(range(200))

dataset = DatasetDict({
    "train": train_dataset,
    "test": test_dataset
})

print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")
print("\nExample:")
print(f"Text: {dataset['train'][0]['text'][:200]}...")
print(f"Label: {'Positive' if dataset['train'][0]['label'] == 1 else 'Negative'}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Training samples: 1000
Test samples: 200

Example:
Text: There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. F...
Label: Positive


## 3. Load Model and Tokenizer

We'll use Qwen2.5-0.5B-Instruct - same as notebook 2.

In [ ]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print(f"Model loaded successfully!")
print(f"Total parameters: {model.num_parameters():,}")

Loading Qwen/Qwen2.5-0.5B-Instruct...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Total parameters: 494,032,768


## 4. Configure LoRA

LoRA will add trainable adapters to the attention layers.

In [ ]:
# LoRA configuration
lora_config = LoraConfig(
    r=8,  # Rank of low-rank matrices
    lora_alpha=16,  # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention layers
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


## 5. Preprocess Dataset

Format reviews as instruction-following examples with labels.

In [ ]:
def preprocess_function(examples):
    """
    Format reviews as chat-style instructions with sentiment labels.
    """
    texts = []
    for text, label in zip(examples["text"], examples["label"]):
        # Truncate long reviews for faster training
        review = text
        sentiment = "positive" if label == 1 else "negative"

        # Format as instruction-response pair
        messages = [
            {"role": "user", "content": f"Classify the sentiment of this movie review as either 'positive' or 'negative'.\n\nReview: {review}"},
            {"role": "assistant", "content": sentiment}
        ]

        text = tokenizer.apply_chat_template(messages, tokenize=False)
        texts.append(text)

    # Tokenize
    model_inputs = tokenizer(
        texts,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # For causal LM, labels are same as input_ids
    model_inputs["labels"] = model_inputs["input_ids"].copy()

    return model_inputs

# Apply preprocessing
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Dataset ready for training!")

Tokenizing dataset...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Dataset ready for training!


## 6. Define Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-qwen-sentiment",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    warmup_steps=50,
    learning_rate=2e-4,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",  # Disable wandb
    push_to_hub=False,
    bf16=True if torch.cuda.is_available() else False
)

# Data collator
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## 7. Train the Model

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

print("Starting LoRA fine-tuning...")
trainer.train()
print("\nTraining complete!")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Starting LoRA fine-tuning...


Epoch,Training Loss,Validation Loss
1,2.781400,2.728167
2,2.798400,2.728034
3,2.766300,2.729394



Training complete!


## 8. Evaluate the Model

Let's test our fine-tuned model on the test set.

In [ ]:
def predict_sentiment(review):
    """
    Predict sentiment for a given review.
    """
    messages = [
        {"role": "user", "content": f"Classify the sentiment of this movie review as either 'positive' or 'negative'.\n\nReview: {review}"}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        temperature=0.3,
        do_sample=True
    )

    prediction = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    return prediction.strip().lower()

print("Testing model on evaluation set...")

Testing model on evaluation set...


## 9. Calculate Accuracy on Test Set

In [ ]:
correct = 0
predictions = []
true_labels = []

# Test on all 200 test examples
for i in range(len(dataset["test"])):
    review = dataset["test"][i]["text"]
    true_label = "positive" if dataset["test"][i]["label"] == 1 else "negative"

    pred = predict_sentiment(review)

    # Check if prediction contains the sentiment
    is_correct = true_label in pred
    if is_correct:
        correct += 1

    predictions.append(1 if "positive" in pred else 0)
    true_labels.append(dataset["test"][i]["label"])

    if i % 20 == 0:
        print(f"Processed {i}/{len(dataset['test'])} examples...")

accuracy = correct / len(dataset["test"]) * 100
print(f"\n{'='*50}")
print(f"LoRA Fine-tuned Model Accuracy: {accuracy:.1f}%")
print(f"Correct predictions: {correct}/{len(dataset['test'])}")

# Calculate additional metrics
precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels, predictions, average='binary'
)
print(f"\nPrecision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")

Processed 0/200 examples...
Processed 20/200 examples...
Processed 40/200 examples...
Processed 60/200 examples...
Processed 80/200 examples...
Processed 100/200 examples...
Processed 120/200 examples...
Processed 140/200 examples...
Processed 160/200 examples...
Processed 180/200 examples...

LoRA Fine-tuned Model Accuracy: 93.0%
Correct predictions: 186/200

Precision: 0.902
Recall: 0.958
F1 Score: 0.929


## 10. Test with Custom Examples

In [ ]:
# Test on custom movie reviews
custom_reviews = [
    "This movie was absolutely fantastic! The acting was superb and I was engaged from start to finish.",
    "Terrible waste of time. Poor acting, weak plot, and boring throughout.",
    "An incredible masterpiece with outstanding cinematography and brilliant performances.",
    "One of the worst movies I've ever seen. Would not recommend to anyone.",
    "It was okay, had some good moments but also some slow parts. Average overall."
]

print("=== Testing Custom Reviews ===\n")
for i, review in enumerate(custom_reviews, 1):
    prediction = predict_sentiment(review)
    print(f"Review {i}: {review}")
    print(f"Predicted Sentiment: {prediction}\n")

=== Testing Custom Reviews ===

Review 1: This movie was absolutely fantastic! The acting was superb and I was engaged from start to finish.
Predicted Sentiment: positive

Review 2: Terrible waste of time. Poor acting, weak plot, and boring throughout.
Predicted Sentiment: negative

Review 3: An incredible masterpiece with outstanding cinematography and brilliant performances.
Predicted Sentiment: positive

Review 4: One of the worst movies I've ever seen. Would not recommend to anyone.
Predicted Sentiment: negative

Review 5: It was okay, had some good moments but also some slow parts. Average overall.
Predicted Sentiment: negative



## 11. Save the LoRA Adapter

In [ ]:
# Save only the LoRA weights (very small!)
model.save_pretrained("./lora-sentiment-adapter")

print("LoRA adapter saved!")
print("\nTo load later:")
print("base_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')")
print("model = PeftModel.from_pretrained(base_model, './lora-sentiment-adapter')")

LoRA adapter saved!

To load later:
base_model = AutoModelForCausalLM.from_pretrained('Qwen/Qwen2.5-0.5B-Instruct')
model = PeftModel.from_pretrained(base_model, './lora-sentiment-adapter')
